# Seasonality Extraction for Commodity Prices

## Learning Objectives

By completing this notebook, you will:
1. Decompose commodity prices into trend, seasonal, and residual components
2. Implement and interpret STL (Seasonal-Trend decomposition using Loess) decomposition
3. Extract Fourier features to model periodic patterns
4. Build custom seasonal kernels for different commodity types
5. Validate seasonal patterns using statistical tests

## Prerequisites
- Time series data handling (Module 2.1)
- Basic Fourier analysis concepts
- Understanding of moving averages

## Estimated Time: 75 minutes

---

In [ ]:
learning_objectives(["Decompose commodity prices into trend, seasonal, and residual components", "Implement and interpret STL (Seasonal-Trend decomposition using Loess) decomposition", "Extract Fourier features to model periodic patterns", "Build custom seasonal kernels for different commodity types", "Validate seasonal patterns using statistical tests", "Time series data handling (Module 2.1)"])

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

apply_course_theme()
apply_plot_theme()

## 1. Setup and Imports

In [ ]:
# Core imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Time series decomposition
from statsmodels.tsa.seasonal import seasonal_decompose, STL
from scipy import signal, stats
from scipy.fft import fft, fftfreq

# Data retrieval
import yfinance as yf
from datetime import datetime, timedelta

# Configuration
np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

print("Environment ready!")

## 2. Why Seasonality Matters for Commodities

**Seasonality** = Predictable patterns that repeat at regular intervals (daily, weekly, monthly, yearly)

**Common seasonal drivers in commodities:**

1. **Natural Gas**: Winter heating demand, summer cooling demand
2. **Crude Oil**: Summer driving season (gasoline), winter heating oil
3. **Agricultural**: Planting cycles, harvest schedules
4. **Electricity**: Daily demand patterns, seasonal temperature effects

**Why extract seasonality?**
- Improves forecast accuracy
- Identifies mispricings (deviations from seasonal norm)
- Separate predictable (seasonal) from random components

## 3. Load and Prepare Commodity Data

In [ ]:
# Fetch natural gas prices (strong seasonal component)
ticker = 'NG=F'
start_date = '2018-01-01'
end_date = '2024-01-01'

print(f"Fetching {ticker} from {start_date} to {end_date}...")
data = yf.download(ticker, start=start_date, end=end_date, progress=False)

# Use adjusted close
prices = data['Adj Close'].dropna()

print(f"Loaded {len(prices)} observations")
print(f"Date range: {prices.index.min()} to {prices.index.max()}")
print(f"\nPrice statistics:")
print(prices.describe())

In [ ]:
# Visualize raw price series
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(prices.index, prices.values, linewidth=1.5, alpha=0.8)
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Price ($/MMBtu)', fontsize=12)
ax.set_title('Natural Gas Futures Prices - Raw Series', fontsize=14)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Classical Additive Decomposition

**Additive model:**
$$y_t = T_t + S_t + R_t$$

where:
- $T_t$ = Trend component (long-term direction)
- $S_t$ = Seasonal component (repeating pattern)
- $R_t$ = Residual (irregular, unpredictable)

**Use additive when:** Seasonal amplitude is constant over time

**Multiplicative model:**
$$y_t = T_t \times S_t \times R_t$$

**Use multiplicative when:** Seasonal amplitude grows with level (common in finance)

In [ ]:
# Classical decomposition (additive)
# Period = 252 (approximate trading days in a year)
decomposition = seasonal_decompose(
    prices, 
    model='additive', 
    period=252,
    extrapolate_trend='freq'
)

# Extract components
trend = decomposition.trend
seasonal = decomposition.seasonal
residual = decomposition.resid

print("Decomposition complete!")
print(f"Trend range: {trend.min():.2f} to {trend.max():.2f}")
print(f"Seasonal range: {seasonal.min():.2f} to {seasonal.max():.2f}")
print(f"Residual std: {residual.std():.2f}")

In [ ]:
# Plot decomposition
fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)

# Original
axes[0].plot(prices.index, prices.values, linewidth=1)
axes[0].set_ylabel('Price', fontsize=11)
axes[0].set_title('Natural Gas: Classical Seasonal Decomposition', fontsize=14)
axes[0].grid(True, alpha=0.3)

# Trend
axes[1].plot(trend.index, trend.values, linewidth=1.5, color='orange')
axes[1].set_ylabel('Trend', fontsize=11)
axes[1].grid(True, alpha=0.3)

# Seasonal
axes[2].plot(seasonal.index, seasonal.values, linewidth=1.5, color='green')
axes[2].set_ylabel('Seasonal', fontsize=11)
axes[2].axhline(0, color='black', linewidth=0.5, linestyle='--')
axes[2].grid(True, alpha=0.3)

# Residual
axes[3].plot(residual.index, residual.values, linewidth=1, color='red', alpha=0.6)
axes[3].set_ylabel('Residual', fontsize=11)
axes[3].set_xlabel('Date', fontsize=12)
axes[3].axhline(0, color='black', linewidth=0.5, linestyle='--')
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**Interpretation:**
- **Trend**: Shows overall price movement over years
- **Seasonal**: Repeating annual pattern (peaks in winter, troughs in summer for nat gas)
- **Residual**: What remains after removing trend and seasonality (should be ~white noise)

## 5. STL Decomposition (More Robust)

**STL** = Seasonal-Trend decomposition using Loess (locally weighted regression)

**Advantages over classical:**
1. Handles any seasonality length
2. Seasonal component can change over time
3. Robust to outliers
4. More flexible trend estimation

In [ ]:
# STL decomposition
stl = STL(
    prices, 
    seasonal=253,  # Seasonal period (trading days/year)
    trend=63,      # Trend window (~3 months)
    robust=True    # Robust to outliers
)

stl_result = stl.fit()

# Extract components
trend_stl = stl_result.trend
seasonal_stl = stl_result.seasonal
residual_stl = stl_result.resid

print("STL decomposition complete!")
print(f"\nVariance explained:")
var_total = prices.var()
var_seasonal = seasonal_stl.var()
var_trend = trend_stl.var()
var_resid = residual_stl.var()
print(f"  Seasonal: {var_seasonal/var_total*100:.1f}%")
print(f"  Trend: {var_trend/var_total*100:.1f}%")
print(f"  Residual: {var_resid/var_total*100:.1f}%")

In [ ]:
# Plot STL decomposition
fig = stl_result.plot()
fig.set_size_inches(14, 12)
plt.suptitle('STL Decomposition - Natural Gas', fontsize=14, y=0.995)
plt.tight_layout()
plt.show()

## 6. Visualizing Seasonal Patterns

Extract and visualize the "typical" seasonal pattern across years.

In [ ]:
# Create seasonal profile (average by month)
seasonal_df = pd.DataFrame({
    'price': prices.values,
    'seasonal': seasonal_stl.values,
    'month': prices.index.month,
    'year': prices.index.year
}, index=prices.index)

# Average seasonal component by month
monthly_seasonal = seasonal_df.groupby('month')['seasonal'].mean()

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
axes[0].bar(range(1, 13), monthly_seasonal.values, color='steelblue', alpha=0.7)
axes[0].set_xticks(range(1, 13))
axes[0].set_xticklabels(month_names)
axes[0].set_xlabel('Month', fontsize=12)
axes[0].set_ylabel('Seasonal Component', fontsize=12)
axes[0].set_title('Average Seasonal Pattern by Month', fontsize=13)
axes[0].axhline(0, color='black', linewidth=1, linestyle='--')
axes[0].grid(True, alpha=0.3, axis='y')

# Line plot over multiple years
for year in seasonal_df['year'].unique():
    year_data = seasonal_df[seasonal_df['year'] == year]
    axes[1].plot(year_data.index.dayofyear, year_data['seasonal'], 
                alpha=0.3, linewidth=1)

# Add average
seasonal_by_doy = seasonal_df.groupby(seasonal_df.index.dayofyear)['seasonal'].mean()
axes[1].plot(seasonal_by_doy.index, seasonal_by_doy.values, 
            color='red', linewidth=2, label='Average')
axes[1].set_xlabel('Day of Year', fontsize=12)
axes[1].set_ylabel('Seasonal Component', fontsize=12)
axes[1].set_title('Seasonal Pattern Across Years', fontsize=13)
axes[1].axhline(0, color='black', linewidth=1, linestyle='--')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**Key Insight:** Natural gas shows strong winter seasonality (heating demand)

## 7. Fourier Features for Seasonality

**Fourier approach:** Represent seasonality as sum of sine/cosine waves

$$S_t = \sum_{k=1}^{K} \left[ a_k \sin\left(\frac{2\pi kt}{P}\right) + b_k \cos\left(\frac{2\pi kt}{P}\right) \right]$$

where:
- $P$ = period (365.25 days for yearly)
- $K$ = number of harmonics (higher $K$ = more flexible)
- Coefficients $a_k, b_k$ learned from data

In [ ]:
def create_fourier_features(index, period, K):
    """
    Create Fourier features for seasonal modeling.
    
    Parameters
    ----------
    index : pd.DatetimeIndex
        Time index
    period : float
        Seasonal period (e.g., 365.25 for yearly)
    K : int
        Number of Fourier pairs (harmonics)
    
    Returns
    -------
    pd.DataFrame
        DataFrame with sin/cos features
    """
    # Time as fraction of period
    t = np.arange(len(index))
    
    features = pd.DataFrame(index=index)
    
    for k in range(1, K + 1):
        features[f'sin_{k}'] = np.sin(2 * np.pi * k * t / period)
        features[f'cos_{k}'] = np.cos(2 * np.pi * k * t / period)
    
    return features

# Create Fourier features
K = 4  # 4 harmonics = flexible seasonality
period = 252  # Trading days per year

fourier_features = create_fourier_features(prices.index, period, K)

print(f"Created {fourier_features.shape[1]} Fourier features")
print("\nFeature names:")
print(fourier_features.columns.tolist())

In [ ]:
# Visualize first few Fourier features
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for i, ax in enumerate(axes.flat):
    col = fourier_features.columns[i]
    ax.plot(fourier_features.index[:500], fourier_features[col].iloc[:500], 
           linewidth=1.5)
    ax.set_title(col, fontsize=12)
    ax.set_xlabel('Date', fontsize=10)
    ax.grid(True, alpha=0.3)

plt.suptitle('Fourier Features (First 500 Days)', fontsize=14)
plt.tight_layout()
plt.show()

## 8. Fit Linear Model with Fourier Features

Use OLS regression to learn Fourier coefficients.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Prepare data
X = fourier_features.values
y = prices.values

# Add time trend
X_with_trend = np.column_stack([np.arange(len(X)), X])

# Fit model
model = LinearRegression()
model.fit(X_with_trend, y)

# Predictions
y_pred = model.predict(X_with_trend)

# Metrics
rmse = np.sqrt(mean_squared_error(y, y_pred))
r2 = r2_score(y, y_pred)

print("Linear model with Fourier features:")
print(f"  R²: {r2:.3f}")
print(f"  RMSE: {rmse:.3f}")
print(f"\nCoefficients:")
print(f"  Trend: {model.coef_[0]:.6f}")
print(f"  Fourier coefficients: {model.coef_[1:]}")

In [ ]:
# Plot fit
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Actual vs fitted
axes[0].plot(prices.index, y, label='Actual', linewidth=1, alpha=0.7)
axes[0].plot(prices.index, y_pred, label='Fitted (Trend + Fourier)', 
            linewidth=1.5, alpha=0.8)
axes[0].set_ylabel('Price', fontsize=12)
axes[0].set_title('Fourier Model Fit', fontsize=14)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Residuals
residuals = y - y_pred
axes[1].plot(prices.index, residuals, linewidth=1, alpha=0.7, color='red')
axes[1].axhline(0, color='black', linewidth=1, linestyle='--')
axes[1].set_ylabel('Residual', fontsize=12)
axes[1].set_xlabel('Date', fontsize=12)
axes[1].set_title('Model Residuals', fontsize=14)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Spectral Analysis: Identifying Dominant Frequencies

Use **Fast Fourier Transform (FFT)** to identify periodic components.

In [ ]:
# Detrend first
from scipy.signal import detrend

y_detrended = detrend(prices.values)

# Compute FFT
N = len(y_detrended)
yf = fft(y_detrended)
xf = fftfreq(N, d=1)[:N//2]  # Only positive frequencies

# Power spectrum
power = 2.0/N * np.abs(yf[:N//2])

# Convert frequency to period
periods = 1 / xf[1:]  # Skip DC component
power = power[1:]

print(f"Computed power spectrum with {len(power)} frequencies")

In [ ]:
# Plot periodogram
fig, ax = plt.subplots(figsize=(14, 6))

# Focus on periods between 10 and 500 days
mask = (periods > 10) & (periods < 500)
ax.plot(periods[mask], power[mask], linewidth=1.5)
ax.set_xlabel('Period (days)', fontsize=12)
ax.set_ylabel('Power', fontsize=12)
ax.set_title('Periodogram - Identifying Seasonal Frequencies', fontsize=14)
ax.axvline(252, color='red', linestyle='--', linewidth=2, 
          label='Annual Period (~252 trading days)')
ax.grid(True, alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()

# Find dominant periods
top_k = 5
top_indices = np.argsort(power[mask])[-top_k:][::-1]
print(f"\nTop {top_k} dominant periods (days):")
for i, idx in enumerate(top_indices, 1):
    print(f"  {i}. {periods[mask][idx]:.1f} days (power: {power[mask][idx]:.2f})")

---

## Exercise 1: Multi-Commodity Seasonal Comparison

**Task:** Compare seasonal patterns across different commodities.

1. Fetch data for at least 3 commodities (e.g., Natural Gas, Crude Oil, Corn)
2. Apply STL decomposition to each
3. Extract monthly seasonal averages
4. Create a plot comparing seasonal profiles

**Question:** Which commodities have strongest seasonality? Why?

In [ ]:
# YOUR CODE HERE
commodities = {
    'NG=F': 'Natural Gas',
    'CL=F': 'Crude Oil',
    'ZC=F': 'Corn'
}

# YOUR CODE: Fetch, decompose, compare
seasonal_profiles = {}

# Store your results in seasonal_profiles dictionary

In [ ]:
# AUTO-GRADED TEST
def test_exercise_1():
    """Verify multi-commodity analysis."""
    assert len(seasonal_profiles) >= 3, "Analyze at least 3 commodities"
    print("✅ Exercise 1 complete! Compare seasonal patterns.")

# test_exercise_1()  # Uncomment after completing

---

## Exercise 2: Optimal Fourier Order Selection

**Task:** Determine optimal number of Fourier harmonics using cross-validation.

1. Split data into train/test (80/20)
2. Try K = 1, 2, ..., 10 harmonics
3. Compute out-of-sample RMSE for each K
4. Plot validation error vs K

**Expected:** Error decreases then plateaus (or increases due to overfitting)

In [ ]:
# YOUR CODE HERE

# Split data
split = int(0.8 * len(prices))
train_idx = prices.index[:split]
test_idx = prices.index[split:]

# YOUR CODE: Cross-validation for K selection
validation_errors = []

# Store RMSE for each K in validation_errors

In [ ]:
# AUTO-GRADED TEST
def test_exercise_2():
    """Verify Fourier order selection."""
    assert len(validation_errors) > 0, "Compute validation errors for different K"
    print("✅ Exercise 2 complete! Optimal K selected.")

# test_exercise_2()  # Uncomment after completing

---

## Exercise 3: Deseasonalized Series Analysis

**Task:** Remove seasonality and analyze remaining variation.

1. Compute deseasonalized series: `y_deseas = y - seasonal_component`
2. Check if deseasonalized series is stationary (ADF test)
3. Compare volatility of original vs deseasonalized
4. Plot ACF/PACF of deseasonalized series

**Goal:** Understand what dynamics remain after removing seasonality

In [ ]:
# YOUR CODE HERE
from statsmodels.tsa.stattools import adfuller

# YOUR CODE: Deseasonalize and analyze
y_deseas = None  # Replace with your deseasonalized series

---

## Summary

### Key Takeaways

1. **Seasonality decomposition** separates predictable patterns from noise
2. **STL** is more flexible and robust than classical decomposition
3. **Fourier features** provide smooth, flexible seasonal representations
4. **Spectral analysis** identifies dominant periodic components
5. **Commodity seasonality** varies by product (heating, harvest, etc.)

### When to Use Each Method

- **Classical decomposition**: Quick exploratory analysis
- **STL**: Production forecasting, handles changing seasonality
- **Fourier**: Regression models, Bayesian priors, when seasonality is smooth
- **Spectral analysis**: Discovering unknown periodicities

### What's Next

Next notebook: **Term Structure Analysis** - Futures curves, contango/backwardation

### Additional Resources

- Cleveland et al. (1990): STL original paper
- Hyndman & Athanasopoulos: *Forecasting: Principles and Practice* (Chapter 3)
- Harvey (1990): *Forecasting, Structural Time Series Models*

---